# 05 - Attention Map Analysis

This notebook visualizes attention patterns for BiLSTM and Transformer models across all gesture classes.

**Goals:**
1. Load trained models from checkpoints
2. Extract attention weights for sample gestures
3. Visualize attention maps per class for both architectures

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src import GestureDataModule, BiLSTMModule, TransformerModule

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

## 1. Load Data and Models

In [ ]:
# Load data
dm = GestureDataModule(batch_size=1)  # Single sample for visualization
dm.setup()

INPUT_SIZE = dm.input_size
CLASS_NAMES = ['air_quotes', 'finger_wagging', 'waving', 'zoom']

print(f"Input size: {INPUT_SIZE}")
print(f"Classes: {CLASS_NAMES}")

In [ ]:
# Load BiLSTM model
bilstm_ckpt = Path('../models/checkpoints/bilstm_best.ckpt')
if bilstm_ckpt.exists():
    bilstm = BiLSTMModule.load_from_checkpoint(bilstm_ckpt, input_size=INPUT_SIZE)
    bilstm.eval()
    bilstm.to(DEVICE)
    print("BiLSTM loaded successfully")
else:
    print(f"BiLSTM checkpoint not found at {bilstm_ckpt}")
    print("Please run 02_training.ipynb first to create the checkpoint")
    bilstm = None

In [ ]:
# Load Transformer model
transformer_ckpt = Path('../models/checkpoints/transformer_best.ckpt')
if transformer_ckpt.exists():
    transformer = TransformerModule.load_from_checkpoint(transformer_ckpt, input_size=INPUT_SIZE)
    transformer.eval()
    transformer.to(DEVICE)
    print("Transformer loaded successfully")
else:
    print(f"Transformer checkpoint not found at {transformer_ckpt}")
    print("Please run 02_training.ipynb first to create the checkpoint")
    transformer = None

## 2. Collect Samples per Class

In [ ]:
# Collect one sample per class for visualization
samples_per_class = {i: None for i in range(4)}

for batch in dm.val_dataloader():
    x, y = batch
    label = y.item()
    if samples_per_class[label] is None:
        samples_per_class[label] = x
    if all(v is not None for v in samples_per_class.values()):
        break

print("Collected samples for all classes:")
for idx, name in enumerate(CLASS_NAMES):
    if samples_per_class[idx] is not None:
        print(f"  {name}: shape {samples_per_class[idx].shape}")

## 3. BiLSTM Attention Visualization

In [ ]:
def visualize_bilstm_attention(model, samples, class_names):
    """Visualize BiLSTM attention weights for each class."""
    if model is None:
        print("Model not loaded")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, (class_idx, sample) in enumerate(samples.items()):
        if sample is None:
            continue
        
        with torch.no_grad():
            sample_gpu = sample.to(DEVICE)
            logits, attention = model.forward_with_attention(sample_gpu)
            pred = logits.argmax(1).item()
            
        if attention is not None:
            # attention shape: (1, seq_len, 1)
            attn_weights = attention.squeeze().cpu().numpy()
            seq_len = len(attn_weights)
            
            ax = axes[idx]
            
            # Plot attention as line
            ax.fill_between(range(seq_len), attn_weights, alpha=0.3, color='blue')
            ax.plot(range(seq_len), attn_weights, 'b-', linewidth=2)
            
            # Mark peak attention
            peak_idx = np.argmax(attn_weights)
            ax.axvline(x=peak_idx, color='red', linestyle='--', alpha=0.7, label=f'Peak: t={peak_idx}')
            
            ax.set_xlabel('Time Step')
            ax.set_ylabel('Attention Weight')
            ax.set_title(f'{class_names[class_idx]}\nPred: {class_names[pred]} ({"✓" if pred == class_idx else "✗"})')
            ax.legend(loc='upper right')
            ax.set_xlim(0, seq_len - 1)
            ax.set_ylim(0, max(attn_weights) * 1.1)
    
    plt.suptitle('BiLSTM Attention Weights per Gesture Class', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../plots/bilstm_attention_maps.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved to plots/bilstm_attention_maps.png")

visualize_bilstm_attention(bilstm, samples_per_class, CLASS_NAMES)

## 4. Transformer Attention Visualization

In [ ]:
def visualize_transformer_attention(model, samples, class_names):
    """Visualize Transformer self-attention patterns for each class."""
    if model is None:
        print("Model not loaded")
        return
    
    n_layers = model.num_layers
    n_classes = len(class_names)
    
    fig, axes = plt.subplots(n_classes, n_layers, figsize=(4 * n_layers, 4 * n_classes))
    if n_layers == 1:
        axes = axes.reshape(-1, 1)
    
    for class_idx, sample in samples.items():
        if sample is None:
            continue
        
        with torch.no_grad():
            sample_gpu = sample.to(DEVICE)
            logits, attention_maps = model.forward_with_attention(sample_gpu)
            pred = logits.argmax(1).item()
        
        for layer_idx, attn in enumerate(attention_maps):
            # attn shape: (1, nhead, seq_len, seq_len)
            # Average across heads
            attn_avg = attn.squeeze(0).mean(dim=0).cpu().numpy()  # (seq_len, seq_len)
            
            ax = axes[class_idx, layer_idx]
            
            sns.heatmap(attn_avg, ax=ax, cmap='Blues', cbar=True, 
                       cbar_kws={'shrink': 0.8}, square=True)
            
            if layer_idx == 0:
                ax.set_ylabel(f'{class_names[class_idx]}\n({"✓" if pred == class_idx else "✗"})', 
                            fontsize=11, fontweight='bold')
            if class_idx == 0:
                ax.set_title(f'Layer {layer_idx + 1}', fontsize=11)
            
            ax.set_xlabel('Key Position')
            if layer_idx == 0:
                ax.set_ylabel(ax.get_ylabel() + '\nQuery Position')
    
    plt.suptitle('Transformer Self-Attention Maps (Averaged Across Heads)', 
                fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../plots/transformer_attention_maps.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved to plots/transformer_attention_maps.png")

visualize_transformer_attention(transformer, samples_per_class, CLASS_NAMES)

## 5. Attention Statistics Summary

In [ ]:
def compute_attention_stats(model, samples, class_names, model_name):
    """Compute attention statistics across samples."""
    if model is None:
        print(f"{model_name} not loaded")
        return
    
    print(f"\n{'='*50}")
    print(f"{model_name} Attention Statistics")
    print('='*50)
    
    for class_idx, sample in samples.items():
        if sample is None:
            continue
        
        with torch.no_grad():
            sample_gpu = sample.to(DEVICE)
            if model_name == 'BiLSTM':
                logits, attention = model.forward_with_attention(sample_gpu)
                if attention is not None:
                    attn = attention.squeeze().cpu().numpy()
                    peak_idx = np.argmax(attn)
                    peak_val = attn[peak_idx]
                    entropy = -np.sum(attn * np.log(attn + 1e-10))
                    
                    print(f"\n{class_names[class_idx]}:")
                    print(f"  Peak attention: t={peak_idx} (weight={peak_val:.4f})")
                    print(f"  Entropy: {entropy:.4f} (lower = more focused)")
                    print(f"  Top-5 attended timesteps: {np.argsort(attn)[-5:][::-1]}")
            else:  # Transformer
                logits, attention_maps = model.forward_with_attention(sample_gpu)
                print(f"\n{class_names[class_idx]}:")
                for layer_idx, attn in enumerate(attention_maps):
                    # Average over heads, then summarize
                    attn_avg = attn.squeeze(0).mean(dim=0).cpu().numpy()
                    diag_mean = np.diag(attn_avg).mean()  # Self-attention
                    off_diag_mean = (attn_avg.sum() - np.trace(attn_avg)) / (attn_avg.size - len(attn_avg))
                    print(f"  Layer {layer_idx + 1}: diag={diag_mean:.4f}, off-diag={off_diag_mean:.4f}")

compute_attention_stats(bilstm, samples_per_class, CLASS_NAMES, 'BiLSTM')
compute_attention_stats(transformer, samples_per_class, CLASS_NAMES, 'Transformer')

## 6. Combined Comparison Plot

In [ ]:
def create_combined_attention_plot(bilstm_model, transformer_model, samples, class_names):
    """Create combined visualization comparing both models."""
    if bilstm_model is None or transformer_model is None:
        print("Both models required for comparison")
        return
    
    fig, axes = plt.subplots(4, 2, figsize=(14, 16))
    
    for class_idx, sample in samples.items():
        if sample is None:
            continue
        
        sample_gpu = sample.to(DEVICE)
        
        # BiLSTM attention
        with torch.no_grad():
            _, bilstm_attn = bilstm_model.forward_with_attention(sample_gpu)
            _, transformer_attn = transformer_model.forward_with_attention(sample_gpu)
        
        ax_bilstm = axes[class_idx, 0]
        ax_transformer = axes[class_idx, 1]
        
        # BiLSTM: line plot
        if bilstm_attn is not None:
            attn = bilstm_attn.squeeze().cpu().numpy()
            ax_bilstm.fill_between(range(len(attn)), attn, alpha=0.3, color='blue')
            ax_bilstm.plot(attn, 'b-', linewidth=2)
            ax_bilstm.set_ylabel(class_names[class_idx], fontsize=12, fontweight='bold')
            ax_bilstm.set_xlabel('Time Step')
            if class_idx == 0:
                ax_bilstm.set_title('BiLSTM Attention', fontsize=12)
        
        # Transformer: heatmap (last layer, averaged heads)
        if transformer_attn:
            attn_avg = transformer_attn[-1].squeeze(0).mean(dim=0).cpu().numpy()
            sns.heatmap(attn_avg, ax=ax_transformer, cmap='Blues', cbar=True, 
                       cbar_kws={'shrink': 0.6}, square=True)
            ax_transformer.set_xlabel('Key Position')
            if class_idx == 0:
                ax_transformer.set_title('Transformer Attention (Last Layer)', fontsize=12)
    
    plt.suptitle('Attention Comparison: BiLSTM vs Transformer', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../plots/attention_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved to plots/attention_comparison.png")

create_combined_attention_plot(bilstm, transformer, samples_per_class, CLASS_NAMES)

In [ ]:
print("\n" + "="*50)
print("Attention Analysis Complete!")
print("="*50)
print("\nGenerated plots:")
print("  - plots/bilstm_attention_maps.png")
print("  - plots/transformer_attention_maps.png")
print("  - plots/attention_comparison.png")